In [ ]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
import os



r_seed = 42

WORKDIR = 'C:/Users/thewr/git/LSTM_giroscopio_acelerometro.git/'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

In [ ]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [ ]:
def combine_csv_data(directory):
    # Specify the directory containing the TXT files

    # Create an empty list to store the DataFrames
    dfs = []

    # Iterate through the TXT files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            # Read the CSV data from the TXT file, assuming no header row
            df = pd.read_csv(os.path.join(directory, filename), header=None)

            # Append the DataFrame to the list
            dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    # Print the combined DataFrame
    return combined_df

In [ ]:
def equalize_dataframe_sizes(df1, df2):
    """Equalizes the sizes of two DataFrames by dropping rows from the larger one.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        tuple: A tuple containing the equalized DataFrames (df1, df2).

    Raises:
        ValueError: If both DataFrames have the same size.
    """

    if df1.shape[0] == df2.shape[0]:
        raise ValueError("Both DataFrames already have the same size.")

    larger_df, smaller_df = (df1, df2) if df1.shape[0] > df2.shape[0] else (df2, df1)

    # Drop rows from the larger DataFrame to match the smaller one
    larger_df = larger_df.iloc[: smaller_df.shape[0], :]

    return larger_df, smaller_df

# Example usage
#df1 = pd.DataFrame({'A': [1, 2, 3, 4, 5], 'B': ['a', 'b', 'c', 'd', 'e']})
#df2 = pd.DataFrame({'A': [10, 20, 30], 'B': ['x', 'y', 'z']})

''' try:
    df1_equal, df2_equal = equalize_dataframe_sizes(df1.copy(), df2.copy())
    #print("Equalized DataFrames:")
    #print(df1_equal)
    #print(df2_equal)
except ValueError as e:
    print(e) '''


In [ ]:
def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Create groups based on unique combinations of X and Y values
    groups_df1 = df1.groupby(['pessoa_gyro','atividade_gyro'])
    groups_df2 = df2.groupby(['pessoa', 'atividade'])

    # Iterate over groups from both DataFrames simultaneously
    for (x, y), group1 in groups_df1:
        #print(group1)
        #if (x,y) in [(1637, 'C'),(1638, 'H'),(1638, 'O'),(1640, 'C'),(1640, 'E')]:
        #    continue
        #else:
        group2 = groups_df2.get_group((x, y))
        '''  group1_has_nan = group1.isnull().sum().sum() > 0
        group2_has_nan = group2.isnull().sum().sum() > 0
        if group1_has_nan == True:
                print(group1.isnull().sum().sum())
                print (group1)
                break
        if group2_has_nan == True:
                print(group2.isnull().sum().sum())
                print (group2)
                break '''
        
        if group2 is not None:  # Check if corresponding group exists in df2
            if group1.shape[0] != group2.shape[0]:
                group1_equal, group2_equal = equalize_dataframe_sizes(group1.copy(), group2.copy())
                #print(f"+++++++++++++++++++++++++++++++++Imprimindo shape grupo 1 e 2 :{group1_equal.shape[0]} - {group2_equal.shape[0]}" )
                #print(f"$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$Imprimindo dados grupo 1 :{group1_equal}" )
                #print("---------------------------------------------")
                #print(f"####################################Imprimindo dados grupo 2 :{group2_equal}" )
            #print(f"+++++++++++++++++++++++++++++++++Imprimindo shape grupo 1 e 2 :{group1_equal.shape[0]} - {group2_equal.shape[0]}" )
            #print(f"$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$Imprimindo dados grupo 1 :{group1_equal}" )
            #print("---------------------------------------------")
            #print(f"####################################Imprimindo dados grupo 2 :{group2_equal}" ) 
            group1_equal = group1_equal.reset_index(drop=True) 
            group2_equal = group2_equal.reset_index(drop=True)  
            df_concat_group1_group2 = pd.concat([group1_equal, group2_equal], axis=1)
            #print(df_concat_group1_group2)
            df_concat_group1_group2 = df_concat_group1_group2.dropna()            
            #print(df_concat_group1_group2)
            has_nan = df_concat_group1_group2.isnull().sum().sum() > 0
            if has_nan == True:
                print (f"**************************** Imprimindo concat co nan: {df_concat_group1_group2}")
                break
            df_concat_group1_group2 = df_concat_group1_group2.reset_index(drop=True) 
            result_df = result_df.reset_index(drop=True)
            result_df = pd.concat([result_df, df_concat_group1_group2])

    return result_df

In [ ]:
''' def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Iterate over consecutive groups in each DataFrame
    for x_val in df1['pessoa'].unique():
        group_df1 = df1[df1['pessoa'] == x_val]
        for y_val in group_df1['atividade'].unique():
            group1 = group_df1[group_df1['atividade'] == y_val]
            group2 = df2[(df2['pessoa'] == x_val) & (df2['atividade'] == y_val)]
            if not group2.empty:  # Check if corresponding group exists in df2
                result_df = pd.concat([result_df, pd.concat([group2,group1], keys=[1, 2], axis=1, ignore_index=True)])

    return result_df '''



In [ ]:
''' # Sample DataFrames
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'C', 'A', 'A', 'B', 'B']})
df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'A', 'B', 'B', 'B', 'B']})

def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Iterate over consecutive groups in each DataFrame
    for x_val in df1['X'].unique():
        group_df1 = df1[df1['X'] == x_val]
        for y_val in group_df1['Y'].unique():
            group1 = group_df1[group_df1['Y'] == y_val]
            group2 = df2[(df2['X'] == x_val) & (df2['Y'] == y_val)]
            if not group2.empty:  # Check if corresponding group exists in df2
                result_df = pd.concat([result_df, pd.concat([group1, group2], keys=[1, 2], axis=1, ignore_index=False)])

    return result_df

# Concatenate the DataFrames
concatenated_df = custom_concat(df1.copy(), df2.copy()) 

print(concatenated_df) '''

In [ ]:
## get dataset

# select only sensor data from smart watches
csv_gyro_dir = WORKDIR +"Data/Raw/watch/gyro"
csv_accel_dir = WORKDIR +"Data/Raw/watch/accel"

In [ ]:
df_csv_gyro = combine_csv_data(csv_gyro_dir)

In [ ]:
df_csv_gyro = df_csv_gyro.replace({';':''}, regex=True)

In [ ]:
df_csv_gyro.columns = ['pessoa_gyro','atividade_gyro','tempo_gyro','giroscópio_x','giroscópio_Y','giroscópio_Z']

In [ ]:
df_csv_gyro.head()

In [ ]:
df_csv_gyro.tail()

In [ ]:
df_csv_gyro.shape

In [ ]:
df_csv_gyro.info()

In [ ]:
condition1 = (df_csv_gyro['pessoa_gyro']== 1637) & (df_csv_gyro['atividade_gyro']=='C')
condition2 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='O')
condition3 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='C')
condition4 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='E')
condition5 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='H')

In [ ]:
#df_csv_gyro_filtered = df_csv_gyro[~(condition1 | condition2 | condition3 | condition4 | condition5)]

In [ ]:
df_csv_gyro_filtered = df_csv_gyro

In [ ]:
df_csv_gyro_filtered.info()

In [ ]:
test = df_csv_gyro_filtered[(df_csv_gyro_filtered['pessoa_gyro']==1638) & (df_csv_gyro_filtered['atividade_gyro']=='H') ]

In [ ]:
print(len(test))

In [ ]:
test2 = df_csv_gyro_filtered[df_csv_gyro_filtered['pessoa_gyro']==1638]

In [ ]:
print(len(test2))

In [ ]:
df_csv_gyro_filtered.shape

In [ ]:
#df_csv_gyro.set_index(["pessoa_gyro","atividade_gyro","tempo_gyro"],inplace=True)

In [ ]:
#df_csv_gyro.head()

In [ ]:
df_csv_gyro.info()

In [ ]:
df_csv_accel = combine_csv_data(csv_accel_dir)

In [ ]:
df_csv_accel = df_csv_accel.replace({';':''}, regex=True)

In [ ]:
df_csv_accel.shape

In [ ]:
df_csv_accel.columns = ['pessoa','atividade','tempo_acelerômetro','acelerômetro_x','acelerômetro_Y','acelerômetro_Z']

In [ ]:
df_csv_accel.head()

In [ ]:
df_csv_accel.tail()

In [ ]:
#df_csv_accel.set_index(["pessoa_acelerômetro","atividade_acelerômetro","tempo_acelerômetro"],inplace=True)

In [ ]:
#df_csv_accel.head()

In [ ]:
condition_accel1 = (df_csv_accel['pessoa']== 1637) & (df_csv_accel['atividade']=='C')
condition_accel2 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='O')
condition_accel3 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='C')
condition_accel4 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='E')
condition_accel5 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='H')

In [ ]:
#df_csv_accel_filtered = df_csv_accel[~(condition_accel1 | condition_accel2 | condition_accel3 | condition_accel4 | condition_accel5)]

In [ ]:
df_csv_accel_filtered = df_csv_accel

In [ ]:
df_csv_accel_filtered.info()

In [ ]:
test3 = df_csv_accel_filtered[(df_csv_accel_filtered['pessoa']==1638) & (df_csv_accel_filtered['atividade']=='H') ]

In [ ]:
print(len(test3))

In [ ]:
test3.head()

In [ ]:
test4 = df_csv_accel_filtered[df_csv_accel_filtered['pessoa']==1638 ]

In [ ]:
print(len(test4))

In [ ]:
df_csv_gyro_filtered.shape

In [ ]:
# Select rows with matching indices (inner join)
''' common_indices = df_csv_accel.index.intersection(df_csv_gyro.index)
df1_filtered = df_csv_accel.loc[common_indices]
df2_filtered = df_csv_gyro.loc[common_indices]

# Concatenate horizontally
result = pd.concat([df1_filtered, df2_filtered], axis=1) '''

In [ ]:
result_concatenation = custom_concat(df_csv_gyro_filtered.copy(),df_csv_accel_filtered.copy())

In [ ]:
result_concatenation.shape

In [ ]:
result_concatenation.head()

In [ ]:
result_concatenation = result_concatenation.reset_index(drop=True)

In [ ]:
result_concatenation.tail()


In [ ]:
test_diff_atividade = result_concatenation[result_concatenation['atividade']!=result_concatenation['atividade_gyro']]

In [ ]:
print(test_diff_atividade.shape[0])

In [ ]:
test_diff_pessoa = result_concatenation[result_concatenation['pessoa']!=result_concatenation['pessoa_gyro']]

In [ ]:
print(len(test_diff_pessoa))

In [ ]:
result_concatenation.columns

In [ ]:
result_concatenation = result_concatenation[['pessoa', 'atividade', 'tempo_acelerômetro', 'acelerômetro_x',
       'acelerômetro_Y', 'acelerômetro_Z', 'tempo_gyro', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']]

In [ ]:
result_concatenation.head()

In [ ]:
result_concatenation = result_concatenation.rename(columns={'pessoa':'pessoa_x'})

In [ ]:
result_concatenation.head()

In [ ]:
# Define the mapping dictionary
mapping_dict = {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'J':9,'K':10,'L':11,'M':12,'O':13,'P':14,'Q':15,'R':16,'S':17}

In [ ]:
result_concatenation["atividade"] = result_concatenation["atividade"].replace(to_replace=mapping_dict)

In [ ]:
result_concatenation.head()

In [ ]:
result_concatenation.tail(50)

In [ ]:
result_concatenation.to_csv('activity_recognition_data_custom_concat_watch.csv', index=False)